# Код для 10 гипотез улучшения baseline

Этот ноутбук — готовый каркас для запуска 10 экспериментов поверх baseline.
Он построен так, чтобы:
- переиспользовать общие функции подготовки данных;
- запускать гипотезы независимо;
- сравнивать их по `WAPE + |Relative Bias|`;
- при желании переводить прогноз в количество машин.

> Ноутбук рассчитан на данные и baseline из соревнования по прогнозу отгрузок.

In [ ]:

from __future__ import annotations

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Iterable

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error

try:
    from catboost import CatBoostRegressor
    CATBOOST_AVAILABLE = True
except Exception:
    CATBOOST_AVAILABLE = False
    CatBoostRegressor = None

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Конфиг
Проверь пути к файлам. По умолчанию я использую те же имена, что и в baseline.

In [ ]:

CONFIG = {
    "train_path": "/mnt/data/train.parquet",   # замени при необходимости
    "test_path": "/mnt/data/test.parquet",     # замени при необходимости
    "forecast_points": 10,                     # 10 шагов по 30 минут = 5 часов
    "time_col": "timestamp",
    "target_col": "target_2h",
    "route_col": "route_id",
    "office_col": "office_from_id",
    "id_col": "id",
    "step_minutes": 30,
    "valid_ratio": 0.2,
    "max_train_rows": None,                    # например 500_000 при нехватке памяти
}

FORECAST_POINTS = CONFIG["forecast_points"]
TIME_COL = CONFIG["time_col"]
TARGET_COL = CONFIG["target_col"]
ROUTE_COL = CONFIG["route_col"]
OFFICE_COL = CONFIG["office_col"]
ID_COL = CONFIG["id_col"]
STEP_MINUTES = CONFIG["step_minutes"]
FUTURE_TARGET_COLS = [f"target_step_{i}" for i in range(1, FORECAST_POINTS + 1)]

## 2. Загрузка данных

In [ ]:

train_path = Path(CONFIG["train_path"])
test_path = Path(CONFIG["test_path"])

if train_path.exists():
    train_df = pd.read_parquet(train_path)
    train_df[TIME_COL] = pd.to_datetime(train_df[TIME_COL])
    train_df = train_df.sort_values([ROUTE_COL, TIME_COL]).reset_index(drop=True)
    print("train shape:", train_df.shape)
    print(train_df.head())
else:
    raise FileNotFoundError(f"Не найден файл train: {train_path}")

if test_path.exists():
    test_df = pd.read_parquet(test_path)
    test_df[TIME_COL] = pd.to_datetime(test_df[TIME_COL])
    test_df = test_df.sort_values([ROUTE_COL, TIME_COL]).reset_index(drop=True)
    print("test shape:", test_df.shape)
    print(test_df.head())
else:
    test_df = None
    print("test file не найден — это нормально, если ты сейчас только валидируешь гипотезы.")

## 3. Базовые списки признаков и метрика

In [ ]:

status_cols = sorted([c for c in train_df.columns if c.startswith("status_")])

class WapePlusRbias:
    @property
    def name(self) -> str:
        return "wape_plus_rbias"

    def calculate(self, y_true: np.ndarray, y_pred: np.ndarray) -> float:
        y_true = np.asarray(y_true, dtype=float)
        y_pred = np.asarray(y_pred, dtype=float)
        denom = np.maximum(y_true.sum(), 1e-9)
        wape = np.abs(y_pred - y_true).sum() / denom
        rbias = abs(y_pred.sum() / denom - 1.0)
        return float(wape + rbias)

metric = WapePlusRbias()

def evaluate_predictions(y_true: pd.DataFrame, y_pred: pd.DataFrame, prefix: str = "") -> pd.DataFrame:
    rows = []
    total_score = metric.calculate(y_true.values, y_pred.values)
    rows.append({"horizon": "all", "metric": total_score})

    for col in y_true.columns:
        rows.append({"horizon": col, "metric": metric.calculate(y_true[col].values, y_pred[col].values)})

    out = pd.DataFrame(rows)
    if prefix:
        out.insert(0, "experiment", prefix)
    return out

## 4. Подготовка supervised-таблицы как в baseline

In [ ]:

def make_future_targets(df: pd.DataFrame, route_col: str = ROUTE_COL, target_col: str = TARGET_COL, horizon: int = FORECAST_POINTS) -> pd.DataFrame:
    df = df.copy()
    grp = df.groupby(route_col, sort=False)
    for step in range(1, horizon + 1):
        df[f"target_step_{step}"] = grp[target_col].shift(-step)
    return df

train_supervised = make_future_targets(train_df)
supervised_df = train_supervised.dropna(subset=FUTURE_TARGET_COLS).copy()

print("supervised shape:", supervised_df.shape)
supervised_df[[ROUTE_COL, TIME_COL, TARGET_COL] + FUTURE_TARGET_COLS[:3]].head()

## 5. Функции feature engineering

Ниже — общий набор функций, из которого собираются все 10 гипотез.

In [ ]:

def add_time_features(df: pd.DataFrame, time_col: str = TIME_COL) -> pd.DataFrame:
    df = df.copy()
    ts = pd.to_datetime(df[time_col])
    df["hour"] = ts.dt.hour
    df["minute"] = ts.dt.minute
    df["dow"] = ts.dt.dayofweek
    df["day"] = ts.dt.day
    df["month"] = ts.dt.month
    df["is_weekend"] = (df["dow"] >= 5).astype(int)
    df["slot_30m"] = df["hour"] * 2 + (df["minute"] >= 30).astype(int)

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["dow_sin"] = np.sin(2 * np.pi * df["dow"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["dow"] / 7)
    df["slot_sin"] = np.sin(2 * np.pi * df["slot_30m"] / 48)
    df["slot_cos"] = np.cos(2 * np.pi * df["slot_30m"] / 48)
    return df


def add_lag_features(
    df: pd.DataFrame,
    group_col: str,
    feature_cols: List[str],
    lags: Iterable[int],
    prefix: str = "lag",
) -> pd.DataFrame:
    df = df.copy()
    grp = df.groupby(group_col, sort=False)
    for col in feature_cols:
        for lag in lags:
            df[f"{col}_{prefix}_{lag}"] = grp[col].shift(lag)
    return df


def add_rolling_features(
    df: pd.DataFrame,
    group_col: str,
    feature_cols: List[str],
    windows: Iterable[int],
    stats: Iterable[str] = ("mean", "std", "max"),
) -> pd.DataFrame:
    df = df.copy()
    grp = df.groupby(group_col, sort=False)

    for col in feature_cols:
        for w in windows:
            rolled = grp[col].rolling(window=w, min_periods=1)
            if "mean" in stats:
                df[f"{col}_rollmean_{w}"] = rolled.mean().reset_index(level=0, drop=True)
            if "std" in stats:
                df[f"{col}_rollstd_{w}"] = rolled.std().reset_index(level=0, drop=True)
            if "max" in stats:
                df[f"{col}_rollmax_{w}"] = rolled.max().reset_index(level=0, drop=True)
            if "min" in stats:
                df[f"{col}_rollmin_{w}"] = rolled.min().reset_index(level=0, drop=True)
    return df


def add_funnel_features(df: pd.DataFrame, status_cols: List[str]) -> pd.DataFrame:
    df = df.copy()
    df["status_sum"] = df[status_cols].sum(axis=1)
    df["status_mean"] = df[status_cols].mean(axis=1)
    df["status_std"] = df[status_cols].std(axis=1)
    df["status_min"] = df[status_cols].min(axis=1)
    df["status_max"] = df[status_cols].max(axis=1)

    for col in status_cols:
        df[f"{col}_share"] = df[col] / np.maximum(df["status_sum"], 1e-6)

    for i in range(len(status_cols) - 1):
        a, b = status_cols[i], status_cols[i + 1]
        df[f"{b}_minus_{a}"] = df[b] - df[a]
        df[f"{b}_div_{a}"] = df[b] / np.maximum(df[a], 1e-6)

    df["late_to_early_ratio"] = (
        df[status_cols[-2:]].sum(axis=1) / np.maximum(df[status_cols[:2]].sum(axis=1), 1e-6)
    )
    return df


def add_hierarchical_aggregates(
    df: pd.DataFrame,
    route_col: str = ROUTE_COL,
    office_col: str = OFFICE_COL,
    target_col: str = TARGET_COL,
    windows: Iterable[int] = (4, 12, 48),
) -> pd.DataFrame:
    df = df.copy()

    # route-level lagged rolling target
    route_grp = df.groupby(route_col, sort=False)[target_col]
    office_grp = df.groupby(office_col, sort=False)[target_col]

    for w in windows:
        df[f"route_target_rollmean_{w}"] = (
            route_grp.shift(1).groupby(df[route_col]).rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
        )
        df[f"office_target_rollmean_{w}"] = (
            office_grp.shift(1).groupby(df[office_col]).rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
        )

    global_route_mean = df.groupby(route_col)[target_col].transform("mean")
    global_office_mean = df.groupby(office_col)[target_col].transform("mean")

    df["route_target_mean_global"] = global_route_mean
    df["office_target_mean_global"] = global_office_mean
    df["route_vs_office_mean_ratio"] = global_route_mean / np.maximum(global_office_mean, 1e-6)
    return df


def add_route_office_rank_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    route_mean = df.groupby(ROUTE_COL)[TARGET_COL].transform("mean")
    office_mean = df.groupby(OFFICE_COL)[TARGET_COL].transform("mean")
    df["route_target_mean"] = route_mean
    df["office_target_mean"] = office_mean
    df["route_mean_rank_in_office"] = (
        df.groupby(OFFICE_COL)["route_target_mean"].rank(method="average", pct=True)
    )
    return df


def add_interaction_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "status_sum" not in df.columns:
        df["status_sum"] = df[status_cols].sum(axis=1)
    if "hour" not in df.columns:
        df = add_time_features(df)
    df["status_sum_x_hour"] = df["status_sum"] * df["hour"]
    df["status_sum_x_weekend"] = df["status_sum"] * df.get("is_weekend", 0)
    if OFFICE_COL in df.columns:
        office_mean_status = df.groupby(OFFICE_COL)["status_sum"].transform("mean")
        df["status_sum_vs_office"] = df["status_sum"] / np.maximum(office_mean_status, 1e-6)
    return df

## 6. Сборка датафрейма признаков под эксперимент

In [ ]:

@dataclass
class FeatureConfig:
    use_time_features: bool = False
    use_status_lags: bool = False
    use_target_lags: bool = False
    use_rollings: bool = False
    use_funnel: bool = False
    use_hierarchy: bool = False
    use_route_office_rank: bool = False
    use_interactions: bool = False

    lag_steps_status: Tuple[int, ...] = (1, 2, 4, 6, 12)
    lag_steps_target: Tuple[int, ...] = (1, 2, 4, 6, 12)
    rolling_windows: Tuple[int, ...] = (2, 4, 8, 12)


def build_feature_table(df: pd.DataFrame, cfg: FeatureConfig) -> pd.DataFrame:
    X = df.copy()

    if cfg.use_time_features:
        X = add_time_features(X)

    if cfg.use_status_lags:
        X = add_lag_features(X, ROUTE_COL, status_cols, cfg.lag_steps_status, prefix="lag")

    if cfg.use_target_lags:
        X = add_lag_features(X, ROUTE_COL, [TARGET_COL], cfg.lag_steps_target, prefix="lag")

    if cfg.use_rollings:
        rolling_base_cols = status_cols.copy()
        if TARGET_COL in X.columns:
            rolling_base_cols = rolling_base_cols + [TARGET_COL]
        X = add_rolling_features(X, ROUTE_COL, rolling_base_cols, cfg.rolling_windows)

    if cfg.use_funnel:
        X = add_funnel_features(X, status_cols)

    if cfg.use_hierarchy:
        X = add_hierarchical_aggregates(X)

    if cfg.use_route_office_rank:
        X = add_route_office_rank_features(X)

    if cfg.use_interactions:
        X = add_interaction_features(X)

    return X

## 7. Train/valid split и подготовка X/y

In [ ]:

def time_split(df: pd.DataFrame, time_col: str = TIME_COL, valid_ratio: float = 0.2) -> Tuple[pd.DataFrame, pd.DataFrame]:
    df = df.sort_values(time_col).copy()
    split_point = df[time_col].quantile(1 - valid_ratio)
    fit_df = df[df[time_col] <= split_point].copy()
    valid_df = df[df[time_col] > split_point].copy()
    return fit_df, valid_df


def prepare_xy(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, List[str]]:
    drop_cols = {TIME_COL, ID_COL, *FUTURE_TARGET_COLS}
    if TARGET_COL in df.columns:
        # target_2h можно оставить как признак, если гипотеза использует lag-based историю.
        # но сам текущий target в модель подавать нельзя для честного online-сценария,
        # поэтому удаляем "сырой" current target.
        drop_cols.add(TARGET_COL)

    feature_cols = [c for c in df.columns if c not in drop_cols]
    X = df[feature_cols].copy()
    y = df[FUTURE_TARGET_COLS].copy()
    return X, y, feature_cols

## 8. Модели

In [ ]:

def make_ridge_model(feature_cols: List[str], X_sample: pd.DataFrame, alpha: float = 3.0):
    categorical = [c for c in feature_cols if c.endswith("_id") or X_sample[c].dtype == "object"]
    numeric = [c for c in feature_cols if c not in categorical]

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), numeric),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
        ]
    )

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", Ridge(alpha=alpha)),
        ]
    )
    return model


def make_hist_gbdt_model():
    base = HistGradientBoostingRegressor(
        loss="absolute_error",
        max_depth=8,
        learning_rate=0.05,
        max_iter=300,
        min_samples_leaf=30,
        random_state=RANDOM_STATE,
    )
    return MultiOutputRegressor(base)


def make_rf_model():
    base = RandomForestRegressor(
        n_estimators=300,
        max_depth=14,
        min_samples_leaf=5,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    return base


def make_catboost_single_model():
    if not CATBOOST_AVAILABLE:
        raise ImportError("CatBoost не установлен в окружении.")
    return CatBoostRegressor(
        loss_function="MAE",
        depth=8,
        learning_rate=0.05,
        iterations=800,
        random_seed=RANDOM_STATE,
        verbose=False,
    )

## 9. Обучение multi-output и per-horizon

In [ ]:

def fit_predict_multioutput(model, X_fit, y_fit, X_valid) -> pd.DataFrame:
    model = clone(model)
    model.fit(X_fit, y_fit)
    pred = model.predict(X_valid)
    pred = np.clip(pred, 0, None)
    return pd.DataFrame(pred, columns=y_fit.columns, index=X_valid.index), model


def fit_predict_per_horizon(
    model_factory,
    X_fit: pd.DataFrame,
    y_fit: pd.DataFrame,
    X_valid: pd.DataFrame,
    cat_features: Optional[List[str]] = None,
) -> Tuple[pd.DataFrame, Dict[str, object]]:
    preds = pd.DataFrame(index=X_valid.index)
    models = {}

    for col in y_fit.columns:
        model = model_factory()
        if CATBOOST_AVAILABLE and isinstance(model, CatBoostRegressor):
            cat_idx = [X_fit.columns.get_loc(c) for c in (cat_features or []) if c in X_fit.columns]
            model.fit(X_fit, y_fit[col], cat_features=cat_idx)
            pred = model.predict(X_valid)
        else:
            model.fit(X_fit, y_fit[col])
            pred = model.predict(X_valid)
        preds[col] = np.clip(pred, 0, None)
        models[col] = model

    return preds, models

## 10. Калибровка под WAPE + |Relative Bias|

In [ ]:

def calibrate_global_scale(y_true: pd.DataFrame, y_pred: pd.DataFrame, grid: Iterable[float] = np.arange(0.8, 1.201, 0.01)):
    best_k = 1.0
    best_score = np.inf
    for k in grid:
        score = metric.calculate(y_true.values, (y_pred.values * k))
        if score < best_score:
            best_score = score
            best_k = float(k)
    calibrated = y_pred * best_k
    return calibrated, {"best_k": best_k, "best_score": best_score}


def calibrate_per_horizon(y_true: pd.DataFrame, y_pred: pd.DataFrame, grid: Iterable[float] = np.arange(0.8, 1.201, 0.01)):
    out = y_pred.copy()
    info = {}
    for col in y_true.columns:
        best_k = 1.0
        best_score = np.inf
        yt = y_true[col].values
        yp = y_pred[col].values
        for k in grid:
            score = metric.calculate(yt, yp * k)
            if score < best_score:
                best_score = score
                best_k = float(k)
        out[col] = y_pred[col] * best_k
        info[col] = {"best_k": best_k, "best_score": best_score}
    return out, info

## 11. Rolling validation для устойчивости

In [ ]:

def make_time_folds(df: pd.DataFrame, n_folds: int = 3, time_col: str = TIME_COL) -> List[Tuple[pd.DataFrame, pd.DataFrame]]:
    uniq_times = np.array(sorted(df[time_col].unique()))
    cut_points = np.linspace(0.55, 0.85, n_folds)
    folds = []
    for frac in cut_points:
        split_idx = max(1, int(len(uniq_times) * frac))
        split_time = uniq_times[split_idx]
        fit_df = df[df[time_col] <= split_time].copy()
        valid_df = df[df[time_col] > split_time].copy()
        if len(valid_df) == 0:
            continue
        folds.append((fit_df, valid_df))
    return folds

## 12. Универсальный запуск эксперимента

In [ ]:

@dataclass
class ExperimentConfig:
    name: str
    feature_cfg: FeatureConfig
    model_kind: str = "ridge"          # ridge | hist_gbdt | rf | catboost_per_horizon
    train_mode: str = "multioutput"    # multioutput | per_horizon
    use_global_calibration: bool = False
    use_horizon_calibration: bool = False
    max_rows: Optional[int] = None


def build_model_for_experiment(model_kind: str, feature_cols: List[str], X_fit: pd.DataFrame):
    if model_kind == "ridge":
        return make_ridge_model(feature_cols, X_fit)
    elif model_kind == "hist_gbdt":
        return make_hist_gbdt_model()
    elif model_kind == "rf":
        return make_rf_model()
    else:
        raise ValueError(f"Для режима {model_kind} используй per_horizon или добавь реализацию.")


def run_experiment(
    base_df: pd.DataFrame,
    exp_cfg: ExperimentConfig,
    valid_ratio: float = CONFIG["valid_ratio"],
) -> Dict[str, object]:
    feat_df = build_feature_table(base_df, exp_cfg.feature_cfg)

    # Удаляем строки, где из-за лагов/роллингов появились NaN.
    feat_df = feat_df.dropna().copy()

    fit_df, valid_df = time_split(feat_df, valid_ratio=valid_ratio)

    if exp_cfg.max_rows is not None and len(fit_df) > exp_cfg.max_rows:
        fit_df = fit_df.sample(exp_cfg.max_rows, random_state=RANDOM_STATE)

    X_fit, y_fit, feature_cols = prepare_xy(fit_df)
    X_valid, y_valid, _ = prepare_xy(valid_df)

    if exp_cfg.train_mode == "multioutput":
        model = build_model_for_experiment(exp_cfg.model_kind, feature_cols, X_fit)
        pred_valid, fitted_model = fit_predict_multioutput(model, X_fit, y_fit, X_valid)
        extra = {"model": fitted_model}
    elif exp_cfg.train_mode == "per_horizon":
        if exp_cfg.model_kind == "catboost_per_horizon":
            cat_features = [c for c in X_fit.columns if c.endswith("_id") or str(X_fit[c].dtype) == "object"]
            pred_valid, models = fit_predict_per_horizon(make_catboost_single_model, X_fit, y_fit, X_valid, cat_features)
        elif exp_cfg.model_kind == "ridge_per_horizon":
            def _factory():
                return make_ridge_model(feature_cols, X_fit)
            pred_valid, models = fit_predict_per_horizon(_factory, X_fit, y_fit, X_valid)
        else:
            raise ValueError(f"Неизвестный per_horizon model_kind: {exp_cfg.model_kind}")
        extra = {"models": models}
    else:
        raise ValueError(f"Неизвестный train_mode: {exp_cfg.train_mode}")

    result = {
        "name": exp_cfg.name,
        "X_fit": X_fit,
        "y_fit": y_fit,
        "X_valid": X_valid,
        "y_valid": y_valid,
        "pred_valid_raw": pred_valid.copy(),
        "pred_valid": pred_valid.copy(),
        "feature_cols": feature_cols,
        **extra,
    }

    if exp_cfg.use_global_calibration:
        calibrated, info = calibrate_global_scale(y_valid, result["pred_valid"])
        result["pred_valid"] = calibrated
        result["global_calibration"] = info

    if exp_cfg.use_horizon_calibration:
        calibrated, info = calibrate_per_horizon(y_valid, result["pred_valid"])
        result["pred_valid"] = calibrated
        result["horizon_calibration"] = info

    result["metrics_raw"] = evaluate_predictions(y_valid, result["pred_valid_raw"], prefix=exp_cfg.name + "_raw")
    result["metrics"] = evaluate_predictions(y_valid, result["pred_valid"], prefix=exp_cfg.name)
    return result

## 13. Базовый baseline для сравнения

In [ ]:

baseline_feature_cfg = FeatureConfig(
    use_time_features=False,
    use_status_lags=False,
    use_target_lags=False,
    use_rollings=False,
    use_funnel=False,
    use_hierarchy=False,
    use_route_office_rank=False,
    use_interactions=False,
)

baseline_exp = ExperimentConfig(
    name="baseline_ridge_simple",
    feature_cfg=baseline_feature_cfg,
    model_kind="ridge",
    train_mode="multioutput",
)

baseline_result = run_experiment(supervised_df, baseline_exp)
display(baseline_result["metrics"])

## 14. 10 гипотез и их код

Ниже каждая гипотеза запускается отдельным объектом `ExperimentConfig`.
Можно выполнять ячейки по одной и сразу сравнивать с baseline.

### H1: лаги статусов и target + rolling

In [ ]:

h1_exp = ExperimentConfig(
    name="h1_lags_rollings_ridge",
    feature_cfg=FeatureConfig(
        use_status_lags=True,
        use_target_lags=True,
        use_rollings=True,
    ),
    model_kind="ridge",
    train_mode="multioutput",
)
h1_result = run_experiment(supervised_df, h1_exp)
display(h1_result["metrics"])

### H2: funnel / flow-признаки по status_1..status_8

In [ ]:

h2_exp = ExperimentConfig(
    name="h2_funnel_ridge",
    feature_cfg=FeatureConfig(
        use_funnel=True,
    ),
    model_kind="ridge",
    train_mode="multioutput",
)
h2_result = run_experiment(supervised_df, h2_exp)
display(h2_result["metrics"])

### H3: календарные и циклические признаки времени

In [ ]:

h3_exp = ExperimentConfig(
    name="h3_time_features_ridge",
    feature_cfg=FeatureConfig(
        use_time_features=True,
    ),
    model_kind="ridge",
    train_mode="multioutput",
)
h3_result = run_experiment(supervised_df, h3_exp)
display(h3_result["metrics"])

### H4: комбинация лагов + funnel + time

In [ ]:

h4_exp = ExperimentConfig(
    name="h4_combo_ridge",
    feature_cfg=FeatureConfig(
        use_time_features=True,
        use_status_lags=True,
        use_target_lags=True,
        use_rollings=True,
        use_funnel=True,
    ),
    model_kind="ridge",
    train_mode="multioutput",
)
h4_result = run_experiment(supervised_df, h4_exp)
display(h4_result["metrics"])

### H5: нелинейная модель HistGradientBoosting

In [ ]:

h5_exp = ExperimentConfig(
    name="h5_hist_gbdt_combo",
    feature_cfg=FeatureConfig(
        use_time_features=True,
        use_status_lags=True,
        use_target_lags=True,
        use_rollings=True,
        use_funnel=True,
        use_interactions=True,
    ),
    model_kind="hist_gbdt",
    train_mode="multioutput",
    max_rows=300_000,
)
h5_result = run_experiment(supervised_df, h5_exp)
display(h5_result["metrics"])

### H6: иерархические агрегаты route/office

In [ ]:

h6_exp = ExperimentConfig(
    name="h6_hierarchy_ridge",
    feature_cfg=FeatureConfig(
        use_time_features=True,
        use_funnel=True,
        use_hierarchy=True,
        use_route_office_rank=True,
    ),
    model_kind="ridge",
    train_mode="multioutput",
)
h6_result = run_experiment(supervised_df, h6_exp)
display(h6_result["metrics"])

### H7: отдельная модель на каждый горизонт

In [ ]:

h7_exp = ExperimentConfig(
    name="h7_ridge_per_horizon",
    feature_cfg=FeatureConfig(
        use_time_features=True,
        use_status_lags=True,
        use_target_lags=True,
        use_rollings=True,
        use_funnel=True,
    ),
    model_kind="ridge_per_horizon",
    train_mode="per_horizon",
)
h7_result = run_experiment(supervised_df, h7_exp)
display(h7_result["metrics"])

### H8: CatBoost по горизонтам

In [ ]:

if CATBOOST_AVAILABLE:
    h8_exp = ExperimentConfig(
        name="h8_catboost_per_horizon",
        feature_cfg=FeatureConfig(
            use_time_features=True,
            use_status_lags=True,
            use_target_lags=True,
            use_rollings=True,
            use_funnel=True,
            use_hierarchy=True,
            use_route_office_rank=True,
            use_interactions=True,
        ),
        model_kind="catboost_per_horizon",
        train_mode="per_horizon",
        max_rows=300_000,
    )
    h8_result = run_experiment(supervised_df, h8_exp)
    display(h8_result["metrics"])
else:
    print("CatBoost недоступен в окружении. Установи пакет и перезапусти эту ячейку.")

### H9: калибровка под WAPE + |Relative Bias|

In [ ]:

h9_exp = ExperimentConfig(
    name="h9_combo_with_global_calibration",
    feature_cfg=FeatureConfig(
        use_time_features=True,
        use_status_lags=True,
        use_target_lags=True,
        use_rollings=True,
        use_funnel=True,
        use_hierarchy=True,
    ),
    model_kind="ridge",
    train_mode="multioutput",
    use_global_calibration=True,
)
h9_result = run_experiment(supervised_df, h9_exp)
display(h9_result["metrics_raw"])
display(h9_result["metrics"])

### H10: калибровка по горизонтам

In [ ]:

h10_exp = ExperimentConfig(
    name="h10_combo_with_horizon_calibration",
    feature_cfg=FeatureConfig(
        use_time_features=True,
        use_status_lags=True,
        use_target_lags=True,
        use_rollings=True,
        use_funnel=True,
        use_hierarchy=True,
    ),
    model_kind="ridge",
    train_mode="multioutput",
    use_horizon_calibration=True,
)
h10_result = run_experiment(supervised_df, h10_exp)
display(h10_result["metrics_raw"])
display(h10_result["metrics"])

## 15. Сравнение всех экспериментов

In [ ]:

all_results = [baseline_result]

for name in ["h1_result","h2_result","h3_result","h4_result","h5_result","h6_result","h7_result","h9_result","h10_result"]:
    if name in globals():
        all_results.append(globals()[name])

if "h8_result" in globals():
    all_results.append(h8_result)

summary_rows = []
for res in all_results:
    score = res["metrics"].query("horizon == 'all'")["metric"].iloc[0]
    summary_rows.append({"experiment": res["name"], "score_all": score})

summary_df = pd.DataFrame(summary_rows).sort_values("score_all")
display(summary_df)

plt.figure(figsize=(12, 5))
plt.bar(summary_df["experiment"], summary_df["score_all"])
plt.xticks(rotation=70, ha="right")
plt.ylabel("WAPE + |Relative Bias|")
plt.title("Сравнение экспериментов")
plt.tight_layout()
plt.show()

## 16. Rolling validation для лучших гипотез

In [ ]:

def run_rolling_validation(base_df: pd.DataFrame, exp_cfg: ExperimentConfig, n_folds: int = 3) -> pd.DataFrame:
    feat_df = build_feature_table(base_df, exp_cfg.feature_cfg).dropna().copy()
    folds = make_time_folds(feat_df, n_folds=n_folds)

    rows = []
    for i, (fit_df, valid_df) in enumerate(folds, start=1):
        if exp_cfg.max_rows is not None and len(fit_df) > exp_cfg.max_rows:
            fit_df = fit_df.sample(exp_cfg.max_rows, random_state=RANDOM_STATE)

        X_fit, y_fit, feature_cols = prepare_xy(fit_df)
        X_valid, y_valid, _ = prepare_xy(valid_df)

        if exp_cfg.train_mode == "multioutput":
            model = build_model_for_experiment(exp_cfg.model_kind, feature_cols, X_fit)
            pred_valid, _ = fit_predict_multioutput(model, X_fit, y_fit, X_valid)
        else:
            if exp_cfg.model_kind == "catboost_per_horizon":
                cat_features = [c for c in X_fit.columns if c.endswith("_id") or str(X_fit[c].dtype) == "object"]
                pred_valid, _ = fit_predict_per_horizon(make_catboost_single_model, X_fit, y_fit, X_valid, cat_features)
            elif exp_cfg.model_kind == "ridge_per_horizon":
                def _factory():
                    return make_ridge_model(feature_cols, X_fit)
                pred_valid, _ = fit_predict_per_horizon(_factory, X_fit, y_fit, X_valid)
            else:
                raise ValueError("train_mode/per_horizon не поддержан для этой модели.")

        if exp_cfg.use_global_calibration:
            pred_valid, _ = calibrate_global_scale(y_valid, pred_valid)
        if exp_cfg.use_horizon_calibration:
            pred_valid, _ = calibrate_per_horizon(y_valid, pred_valid)

        score = metric.calculate(y_valid.values, pred_valid.values)
        rows.append({
            "experiment": exp_cfg.name,
            "fold": i,
            "score": score,
            "fit_rows": len(fit_df),
            "valid_rows": len(valid_df),
        })
    return pd.DataFrame(rows)

# Пример: rolling validation для лучшей гипотезы
best_exp_cfg = h10_exp if "h10_exp" in globals() else h4_exp
rolling_scores = run_rolling_validation(supervised_df, best_exp_cfg, n_folds=3)
display(rolling_scores)
display(rolling_scores.groupby("experiment", as_index=False)["score"].agg(["mean", "std"]).reset_index())

## 17. Ансамбль лучших моделей

In [ ]:

def blend_predictions(preds_with_weights: List[Tuple[pd.DataFrame, float]]) -> pd.DataFrame:
    total_weight = sum(w for _, w in preds_with_weights)
    out = None
    for pred, w in preds_with_weights:
        cur = pred * w
        out = cur if out is None else out.add(cur, fill_value=0.0)
    return out / total_weight

blend_candidates = []
for name, weight in [("h4_result", 1.0), ("h5_result", 1.0), ("h6_result", 1.0), ("h10_result", 1.25)]:
    if name in globals():
        blend_candidates.append((globals()[name]["pred_valid"], weight))

if blend_candidates:
    blend_pred = blend_predictions(blend_candidates)
    blend_score = evaluate_predictions(baseline_result["y_valid"], blend_pred, prefix="blend")
    display(blend_score)
else:
    print("Сначала запусти несколько гипотез, чтобы собрать ансамбль.")

## 18. Перевод прогноза в количество машин
Это уже слой бизнес-логики для командного трека.

In [ ]:

def convert_volume_to_cars(
    forecast_df: pd.DataFrame,
    volume_col: str = "y_pred",
    capacity_per_car: float = 33.0,
    safety_coef: float = 1.05,
    min_cars: int = 0,
) -> pd.DataFrame:
    out = forecast_df.copy()
    adjusted_volume = np.maximum(out[volume_col], 0) * safety_coef
    out["cars_needed"] = np.maximum(np.ceil(adjusted_volume / capacity_per_car), min_cars).astype(int)
    return out

demo_forecast = pd.DataFrame({
    "route_id": [1, 1, 2],
    "timestamp": pd.to_datetime(["2026-01-01 10:00:00", "2026-01-01 10:30:00", "2026-01-01 10:00:00"]),
    "y_pred": [17.5, 40.2, 68.4],
})

display(convert_volume_to_cars(demo_forecast, capacity_per_car=33, safety_coef=1.10))

## 19. Шаблон инференса для лучшей модели
Ниже — заготовка под baseline-логику: обучить на train и сделать прогноз на тестовый горизонт.

In [ ]:

def forecast_from_last_fact(best_result: Dict[str, object], base_train_df: pd.DataFrame, test_df: pd.DataFrame) -> pd.DataFrame:
    inference_ts = base_train_df[TIME_COL].max()
    latest_fact_df = base_train_df[base_train_df[TIME_COL] == inference_ts].copy()

    # нужно построить признаки в том же стиле, что и для обучения
    # пример ниже — для h10, но можно подставить любой exp_cfg
    exp_map = {
        "baseline_ridge_simple": baseline_exp,
    }
    for nm in ["h1_exp","h2_exp","h3_exp","h4_exp","h5_exp","h6_exp","h7_exp","h8_exp","h9_exp","h10_exp"]:
        if nm in globals():
            exp_map[globals()[nm].name] = globals()[nm]

    exp_cfg = exp_map[best_result["name"]]

    # ВАЖНО: признаки со сдвигами считаются на всей train-истории, затем берется последний timestamp
    feat_train = build_feature_table(make_future_targets(base_train_df), exp_cfg.feature_cfg)
    feat_latest = feat_train[feat_train[TIME_COL] == inference_ts].dropna().copy()

    X_test = feat_latest[best_result["feature_cols"]].copy()

    if "model" in best_result:
        pred = best_result["model"].predict(X_test)
        pred = pd.DataFrame(np.clip(pred, 0, None), columns=FUTURE_TARGET_COLS, index=X_test.index)
    elif "models" in best_result:
        pred = pd.DataFrame(index=X_test.index)
        for col, model in best_result["models"].items():
            pred[col] = np.clip(model.predict(X_test), 0, None)
    else:
        raise ValueError("В result не найдено обученной модели.")

    if "global_calibration" in best_result:
        pred = pred * best_result["global_calibration"]["best_k"]

    if "horizon_calibration" in best_result:
        for col, info in best_result["horizon_calibration"].items():
            pred[col] = pred[col] * info["best_k"]

    pred["route_id"] = feat_latest[ROUTE_COL].values

    forecast = pred.melt(
        id_vars="route_id",
        value_vars=FUTURE_TARGET_COLS,
        var_name="step",
        value_name="y_pred"
    )
    forecast["step_num"] = forecast["step"].str.extract(r"(\d+)").astype(int)
    forecast["timestamp"] = inference_ts + pd.to_timedelta(forecast["step_num"] * STEP_MINUTES, unit="m")
    forecast = forecast[[ROUTE_COL, TIME_COL, "y_pred"]].sort_values([ROUTE_COL, TIME_COL]).reset_index(drop=True)

    if test_df is not None and ID_COL in test_df.columns:
        submission = test_df.merge(forecast, on=[ROUTE_COL, TIME_COL], how="left")[[ID_COL, "y_pred"]]
        return submission
    return forecast

# пример:
# best_result = h10_result
# submission = forecast_from_last_fact(best_result, train_df, test_df)
# submission.to_csv("submission_best.csv", index=False)
# display(submission.head())

## 20. Журнал экспериментов

In [ ]:

experiment_log = pd.DataFrame([
    {
        "experiment": res["name"],
        "score_all": res["metrics"].query("horizon == 'all'")["metric"].iloc[0],
        "n_features": len(res["feature_cols"]),
        "fit_rows": len(res["X_fit"]),
        "valid_rows": len(res["X_valid"]),
    }
    for res in all_results
]).sort_values("score_all")

display(experiment_log)